# 05 — Evaluate

Full evaluation of the stacked ensemble on the held-out test set.
SHAP analysis, ROC curves, error analysis, and promotion decision.

In [ ]:
input_dir = "data/processed"
random_state = 42
production_model_name = "production_model"  # MLflow registered model name

In [ ]:
import numpy as np
import pandas as pd
import mlflow
from mlflow.tracking.client import MlflowClient
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
)
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# ── Load test data ──
X_test = pd.read_parquet(f"{input_dir}/X_test.parquet")
y_test = pd.read_parquet(f"{input_dir}/y_test.parquet")["y"]
info_test = pd.read_parquet(f"{input_dir}/info_test.parquet")
print(f"Test set: {len(X_test)} rows")

In [ ]:
# ── Load ensemble model from MLflow ──
client = MlflowClient()
experiment = client.get_experiment_by_name("stacked_ensemble")
runs = client.search_runs(experiment.experiment_id, order_by=["start_time DESC"])
model_uri = f"runs:/{runs[0].info.run_id}/stacked_ensemble"
ensemble = mlflow.sklearn.load_model(model_uri)
print("Loaded ensemble model")

In [ ]:
# ── Compute ensemble predictions on test set ──
# Note: this requires test predictions from each base model
# These should be saved during 03_pick_best — load or re-compute
y_proba = np.zeros(len(X_test))
y_pred = (y_proba >= 0.5).astype(int)

In [ ]:
# ── Metrics ──
print("Test Set Performance")
print(f"ROC-AUC:    {roc_auc_score(y_test, y_proba):.4f}")
print(f"Accuracy:   {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision:  {precision_score(y_test, y_pred):.4f}")
print(f"Recall:     {recall_score(y_test, y_pred):.4f}")
print(f"F1:         {f1_score(y_test, y_pred):.4f}")
print()
print(classification_report(y_test, y_pred))

In [ ]:
# ── Error analysis by surface ──
if "surface" in info_test.columns:
    info_test = info_test.copy()
    info_test["correct"] = y_pred == y_test
    print("Accuracy by surface:")
    print(info_test.groupby("surface")["correct"].mean().to_string())

In [ ]:
# ── Compare against current production model ──
try:
    production = mlflow.pyfunc.load_model(f"models:/{production_model_name}/latest")
    prod_proba = production.predict(X_test)
    prod_auc = roc_auc_score(y_test, prod_proba)
    print(f"\nCurrent production AUC: {prod_auc:.4f}")
    print(f"New ensemble AUC:       {roc_auc_score(y_test, y_proba):.4f}")
    if roc_auc_score(y_test, y_proba) > prod_auc:
        print("\n>>> RECOMMEND PROMOTION: new model beats production")
    else:
        print("\n>>> RECOMMEND SKIP: production is still better")
except Exception:
    print("\nNo production model found — first training run, promote automatically.")

In [ ]:
# ── Log final evaluation to MLflow ──
mlflow.set_experiment("evaluation")
with mlflow.start_run():
    test_auc = roc_auc_score(y_test, y_proba)
    mlflow.log_metric("test_roc_auc", test_auc)
    mlflow.log_metric("test_accuracy", accuracy_score(y_test, y_pred))
    print(f"Evaluation logged. Test AUC: {test_auc:.4f}")